# Timing-driven placement in fifteen lines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RayChang987/Attar/blob/main/examples/colab/timing_driven_colab.ipynb)

The README demo of [Attar](https://github.com/RayChang987/Attar), cell by cell: place an ASAP7 design (cva6, 121k
cells, 8 SRAMs) for wirelength, time it, weight the nets by their slack the way OpenROAD's timing-driven global
placement does, place again, time it again.

**Runtime -> Change runtime type -> T4 GPU** (it also runs on the CPU, only slower).

## 0. Setup

In [ ]:
!pip install -q attar==1.0.5

In [ ]:
import torch
import matplotlib.pyplot as plt
from IPython.display import Image
import attar

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('attar', attar.__version__, 'on', torch.cuda.get_device_name() if dev == 'cuda' else 'CPU')


def show(png):                            # a plot d.plot() wrote, inline
    plt.close('all')
    return Image(png)

## 1. Load and place for wirelength

`attar.load` fetches the design from the [Attar zoo](https://huggingface.co/datasets/RayChang987/attar-zoo) on
Hugging Face (a few MB, with its ASAP7 timing library) and puts it on the GPU as tensors. `d.place()` is global
placement plus legalization.

In [ ]:
d = attar.load('after_fp/orfs_asap7/cva6', device=dev)
d.place()                                 # wirelength only
hpwl0 = float(d.hpwl())
d.plot('wirelength.png')
show('wirelength.png')

## 2. Time it

Static timing as OpenSTA computes it, with placement parasitics as OpenROAD estimates them.

The **first timing run in a fresh runtime compiles the GPU timer's Triton kernels: about two minutes**. Every
later run takes about a second.

In [ ]:
before = d.timing_report(endpoints=0, path=False)

## 3. Weight the nets by their slack

`net_slack()` gives every net's setup slack, as OpenSTA's `Sta::netSlack` (infinite for unconstrained nets).

In [ ]:
s = d.timing().net_slack()                # every net, as OpenSTA
print('%d nets with a slack, %d of them negative; worst %.1f ps' % (s.isfinite().sum(), (s < 0).sum(), float(s.min())))

OpenROAD's `gpl` weights the worst 10 % of the nets, ramping their weight from 1 to 5 by criticality. The weights
are a tensor, `d.n_weight`; the next placement reads them. This is the part to change for your own method: any
weighting, any formula.

In [ ]:
v = s[s.isfinite()].sort().values
v = v[:len(v) // 10]                      # the worst 10 %
lo, hi = v[len(v) // 50], v[-1]           # the ramp: weight 5 at lo and below, 1 at hi
w = d.n_weight
d.n_weight = w * (1 + 4 * ((hi - s) / (hi - lo)).clamp(0, 1))
print('%d nets weighted, from %.1f ps (weight 5) to %.1f ps (weight 1)' % (len(v), float(lo), float(hi)))

## 4. Place again, timing-driven, and time it

In [ ]:
d.place(verbose=False)                    # timing-driven
d.n_weight = w                            # back to the netlist's weights (HPWL, plots)
after = d.timing_report(endpoints=0, path=False)
d.plot('timing_driven.png')
show('timing_driven.png')

In [ ]:
print('             %12s %12s' % ('wirelength', 'timing-driven'))
print('WNS (ps)     %12.1f %12.1f' % (before['wns'], after['wns']))
print('TNS (ps)     %12.1f %12.1f' % (before['tns'], after['tns']))
print('violations   %12d %12d' % (before['setup_violations'], after['setup_violations']))
print('HPWL (um)    %12.0f %12.0f  (%+.1f%%)' % (hpwl0, float(d.hpwl()), 100 * (float(d.hpwl()) / hpwl0 - 1)))
print('legal:', attar.Evaluator(d).legality()['legal'])

## 5. Take it out

`d.export` writes the placement into the design's own DEF (die, rows, PDN, macros kept), ready for OpenROAD to
route and sign off.

In [ ]:
d.export('cva6_timing_driven.def');
# from google.colab import files; files.download('cva6_timing_driven.def')

## Next

* Another design: `attar.zoo('after_fp', remote=True)` lists them, e.g. `after_fp/orfs_asap7/swerv` (the second
  design in the README demo).
* Timing-driven placement research on Attar, step by step:
  [net and pin weighting on superblue18](https://colab.research.google.com/github/RayChang987/Attar/blob/main/examples/colab/p11_attar_superblue18.ipynb)
  and [LATTE, timing-driven detailed placement](https://colab.research.google.com/github/RayChang987/Attar/blob/main/examples/colab/latte_on_attar.ipynb).
* Your own weighting, or your own placer on `d.pos`: the [API](https://github.com/RayChang987/Attar/blob/main/docs/api.md)
  and the [examples](https://github.com/RayChang987/Attar/tree/main/examples).